In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True,
    download=True, transform=transform
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False,
    download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

100.0%
100.0%
100.0%
100.0%


In [3]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1    = nn.Conv2d(1, 16, kernel_size=5, padding=2)
        self.conv2    = nn.Conv2d(16, 32, kernel_size=5, padding=2)
        self.pool     = nn.MaxPool2d(2, 2)
        self.fc1      = nn.Linear(32 * 7 * 7, 128)
        self.fc2      = nn.Linear(128, 10)
        self.dropout  = nn.Dropout(p=0.25)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

In [4]:
model     = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [6]:
n_epochs = 3

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{n_epochs} — Loss: {running_loss/len(train_loader):.4f}")

Epoch 1/3 — Loss: 0.2378
Epoch 2/3 — Loss: 0.2127
Epoch 3/3 — Loss: 0.1957


In [7]:
model.eval()
correct = 0
total   = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs    = model(images)
        _, predicted = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 91.39%
